# MH-05. Diagnostic + descriptive tables

Diagnostics on all candidate nodes (no floor); descriptive on the ready-to-run cohort.
Includes later-psychosis onset (first 295.x dated > index; single occurrence, not rule-of-two)
with raw counts at 6mo/1y/2y and KM cumulative incidence (censor at post_index_observation_days).


## 1. Load

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_DIR = Path('output/mh')
spine_all = pd.read_parquet(OUTPUT_DIR / 'spine.parquet')
Y = pd.read_parquet(OUTPUT_DIR / 'Y_nodes.parquet').set_index('person_id')
mh_dx = pd.read_parquet(OUTPUT_DIR / 'mh_dx_events.parquet')
node_table = pd.read_parquet(OUTPUT_DIR / 'mh_node_table.parquet')
build = OUTPUT_DIR / 'ready_to_run'
spine_ready = pd.read_parquet(build / 'spine.parquet') if (build/'spine.parquet').exists() else spine_all


## 2. Inclusion funnel + category counts + age distribution

In [ ]:
funnel = {
    'all_mh_patients_18_35_index': int(len(spine_all)),
    'ready_to_run_after_lookback': int(len(spine_ready)),
}
print('inclusion funnel:', funnel)
print('\ncounts by MH category (index_signal_type):')
print(spine_ready['index_signal_type'].value_counts().sort_index())
print('\nage at index:')
print(spine_ready['age_at_index'].describe())


## 3. Per-node pre/at/post-index split + first==index share

In [ ]:
dx = mh_dx.merge(spine_all[['person_id','index_date']], on='person_id')
dx['dx_day'] = pd.to_datetime(dx['dx_date']).dt.normalize()
dx['index_date'] = pd.to_datetime(dx['index_date'])
dx['rel'] = np.select([dx['dx_day'] < dx['index_date'],
                       dx['dx_day'] == dx['index_date']],
                      ['pre','at'], default='post')
split = dx.groupby(['phecode','rel']).size().unstack(fill_value=0)
print('per-node pre/at/post (head):')
print(split.head(15))
split.to_csv(OUTPUT_DIR / 'diag_node_pre_at_post.csv')


## 4. Later-psychosis onset: counts + KM

In [ ]:
# first 295.x strictly AFTER index (single dated occurrence, not rule-of-two).
psy_nodes = set(node_table[node_table['is_psychosis']]['phecode'])
psy = dx[dx['phecode'].isin(psy_nodes) & (dx['dx_day'] > dx['index_date'])]
first_psy = psy.groupby('person_id')['dx_day'].min().rename('psy_date')

base = spine_ready[['person_id','index_date','post_index_observation_days']].copy()
base['index_date'] = pd.to_datetime(base['index_date'])
base = base.merge(first_psy, on='person_id', how='left')
base['t_event'] = (base['psy_date'] - base['index_date']).dt.days
base['event'] = base['psy_date'].notna().astype(int)
base['t'] = np.where(base['event'] == 1, base['t_event'], base['post_index_observation_days'])

for horizon, days in [('6mo',182),('1y',365),('2y',730)]:
    n = int(((base['event'] == 1) & (base['t_event'] <= days)).sum())
    print(f'later-psychosis within {horizon}: raw count = {n}')

try:
    from lifelines import KaplanMeierFitter
    kmf = KaplanMeierFitter()
    kmf.fit(base['t'].clip(lower=0), base['event'], label='later_psychosis')
    for days in [182,365,730]:
        ci = 1 - float(kmf.survival_function_at_times(days).iloc[0])
        print(f'KM cumulative incidence at {days}d: {ci:.4f}')
except ImportError:
    print('lifelines not installed -> pip install lifelines for KM cumulative incidence')

base[['person_id','index_date','t','event']].to_csv(OUTPUT_DIR / 'later_psychosis_survival.csv', index=False)


## 5. Patient table

In [ ]:
patient_tbl = spine_ready[['person_id','index_date','age_at_index','index_signal_type',
                            'index_concept_label','post_index_observation_days']].copy()
patient_tbl = patient_tbl.merge(base[['person_id','event']].rename(columns={'event':'later_psychosis'}),
                                on='person_id', how='left')
patient_tbl.to_csv(OUTPUT_DIR / 'patient_table.csv', index=False)
print('patient table:', patient_tbl.shape)
patient_tbl.head()
